In [ ]:
import os
import numpy as np
from neo4j import GraphDatabase
import networkx as nx
from sentence_transformers import SentenceTransformer
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from sklearn.preprocessing import normalize
import pickle
import faiss


# Configuration
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = os.getenv("NEO4J_PASS", "adminadmin")
# node properties with textual info for embedding
TEXT_FIELDS = ["code", "context", "feature_name","function_name","instance","name","runtime_exceptions","type","value","id"]


# Node2Vec params
NODE2VEC_DIM = 128
NODE2VEC_P = 1
NODE2VEC_Q = 1
WALK_LENGTH = 30
NUM_WALKS = 10

# Text embedding model
TEXT_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2" 

GEN_MAX_TOKENS = 256




def connect_neo4j():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS), notifications_disabled_categories=["DEPRECATION"])
    return driver

def fetch_graph(driver):
    """
    Fetch nodes and relationships from Neo4j.
    Expects nodes to have an 'id' property or uses internal id.
    Returns:
        nodes: {node_id: {labels: [...], props: {...}}}
        edges: list of (src_id, dst_id, rel_type)
    """
    with driver.session() as session:
        # fetch nodes with their internal id, labels and properties
        nodes_res = session.run("MATCH (n) RETURN id(n) as id, labels(n) as labels, properties(n) as props")
        nodes = {}
        for r in nodes_res:
            nid = r["id"]
            nodes[nid] = {"labels": r["labels"], "props": r["props"]}
        # fetch relationships
        rels_res = session.run("MATCH (a)-[r]->(b) RETURN id(a) as a, id(b) as b, type(r) as t")
        edges = []
        for r in rels_res:
            edges.append((r["a"], r["b"], r["t"]))
    return nodes, edges


# Build structural embeddings via node2vec

def compute_node2vec_embeddings(driver, dim=NODE2VEC_DIM):
    """
    Compute structural embeddings using Neo4j Graph Data Science (GDS) node2vec.
    Stores embeddings back into Neo4j and returns a dict {node_id: np.array}.
    """

    # Create a GDS in-memory projection
    with driver.session() as session:
        # ensures the projection doesn't already exist from a failed run
        session.run("CALL gds.graph.drop('provGraph', false)")
        #if not creates it
        session.run("""
        CALL gds.graph.project(
            'provGraph',
            '*',          // all node labels
            '*'           // all relationship types
        )
        """)

    # Run node2vec and WRITE the results back to Neo4j 
    # this is the "structural" part of the embedding
    with driver.session() as session:
        # Use gds.node2vec.write (does the same thing as mutate but also allows saving)
        session.run(f"""
        CALL gds.node2vec.write('provGraph', {{
            embeddingDimension: {dim},
            writeProperty: 'node2vec_emb',  // specify the property name to save
            walkLength: 40,
            iterations: 5,
            inOutFactor: 1.0,
            returnFactor: 1.0,
            windowSize: 5,
            concurrency: 2
        }})
        YIELD nodePropertiesWritten, computeMillis
        """)

    # Extract embeddings back into Python
    embeddings = {}
    with driver.session() as session:
        result = session.run("MATCH (n) WHERE n.node2vec_emb IS NOT NULL RETURN id(n) as id, n.node2vec_emb as emb")
        for record in result:
            nid = record["id"]
            emb = np.array(record["emb"], dtype=np.float32)
            embeddings[nid] = emb

    # Drop the in-memory projection to free resources
    with driver.session() as session:
        session.run("CALL gds.graph.drop('provGraph', false)")
    
    # Clean up the property from your graph
    with driver.session() as session:
        session.run("MATCH (n) WHERE n.node2vec_emb IS NOT NULL REMOVE n.node2vec_emb")
        print("Cleaned up node2vec_emb property from Neo4j.")

    return embeddings



# Build text embeddings

def compute_text_embeddings(nodes, model_name=TEXT_EMBED_MODEL):
    st = SentenceTransformer(model_name)
    text_emb = {}
    for nid, node in nodes.items():
        # build a textual summary from selected properties
        parts = []
        for f in TEXT_FIELDS:
            if f in node["props"] and node["props"][f]:
                parts.append(str(node["props"][f]))
        # include labels as short context
        parts.append("labels: " + ",".join(node["labels"]))
        text = " | ".join(parts) if parts else f"node_{nid}"
        vec = st.encode(text)
        text_emb[nid] = {"vec": vec, "text": text}
    return text_emb


# Create combined vectors (hybrid through concatenation of textual and structural embedding)
# and index in FAISS

def build_and_store_vector_index(nodes, node2vec_emb, text_emb, index_path="faiss_index"):
    # Combine embeddings, normalize each part then concatenate
    ids = []
    vectors = []
    metadatas = []
    for nid in nodes.keys():
        
        # Check if a structural embedding exists for this node.
        # If not, it's likely an isolated node (the node2vec could not be created), so skips it.
        if nid not in node2vec_emb:
            print(f"Skipping node {nid} as it has no structural embedding.") #in most cases useless but makes the code more robust
            continue

        s_vec = node2vec_emb[nid]
        t_vec = text_emb[nid]["vec"]
        # normalize
        s_vec = normalize(s_vec.reshape(1, -1))[0]
        t_vec = normalize(t_vec.reshape(1, -1))[0]
        # weights. How much influence to give to the structural vs semantic part of the embedding
        STRUCT_W = 0.4
        SEM_W = 0.6
        combined = np.concatenate([STRUCT_W * s_vec, SEM_W * t_vec])
        ids.append(str(nid))
        vectors.append(combined.astype("float32"))
        metadatas.append({
            "node_id": str(nid),
            "text": text_emb[nid]["text"],
            "labels": nodes[nid]["labels"],
            "props": nodes[nid]["props"]
        })

    dim = vectors[0].shape[0]
    xb = np.vstack(vectors)
    index = faiss.IndexFlatIP(dim)  # inner product
    # normalize for ip search (L2)
    faiss.normalize_L2(xb)
    index.add(xb)
    # store metadata mapping separately
    meta_path = os.path.join(index_path, "meta.pkl")
    os.makedirs(index_path, exist_ok=True)
    with open(meta_path, "wb") as f:
        pickle.dump({"ids": ids, "metadatas": metadatas, "xb": xb}, f)
    # also pickle faiss index
    faiss.write_index(index, os.path.join(index_path, "faiss.idx"))
    print(f"Saved index and metadata to {index_path}")
    return os.path.join(index_path, "faiss.idx"), meta_path


# Load index + perform similarity search (nearest neighbors via FAISS)

def load_index_and_search(index_path, meta_path, query_vec, top_k=5):
    index = faiss.read_index(index_path)
    # load metadata
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)
    # normalize query
    q = query_vec.reshape(1, -1).astype("float32")
    faiss.normalize_L2(q)
    D, I = index.search(q, top_k) #top k most similar verctors
    hits = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        hits.append({"score": float(score), "id": meta["ids"][idx], "meta": meta["metadatas"][idx]})
    return hits


# Build query embedding (semantic) and map to combined space

def embed_query_semantic(query, text_model=TEXT_EMBED_MODEL, target_struct_dim=NODE2VEC_DIM, struct_weight=0.4, sem_weight=0.6):
    st = SentenceTransformer(text_model)
    q_sem = st.encode(query)
    q_sem = normalize(q_sem.reshape(1, -1))[0]
    # For structural portion we don't have a direct structural vector for query.
    # For the prototype: use zeros for structural part, but weight appropriately. MAYBE CHANGE THIS PART TALK ABOUT IT NEXT UPDATE MEETING
    s_part = np.zeros((target_struct_dim,))
    combined = np.concatenate([struct_weight * s_part, sem_weight * q_sem])
    return combined.astype("float32")


# Cypher expansion: for each retrieved node, expand (2 but can be customized) hops to collect provenance context

def cypher_expand_context(driver, node_id, hops=2, include_types=None):
    """
    Returns a list of paths (as textual bullet points) around the node.
    """
    include_clause = ""
    if include_types:
        #customization
        # e.g. only include Activity nodes etc. Not enforced here; TALK ABOUT POSSIBILITIES NEXT UPDATE MEETING
        pass
    cypher = f"""
    MATCH (n)
    WHERE id(n) = $nid
    CALL apoc.path.subgraphAll(n, {{maxLevel: {hops}}})
    YIELD nodes, relationships
    RETURN [x IN nodes | {{id: id(x), labels: labels(x), props: properties(x)}}] as nodes,
           [r IN relationships | {{start: id(startNode(r)), end: id(endNode(r)), type: type(r)}}] as rels
    """
    # Uses APOC to call the query and save the bullet points(text)
    with driver.session() as session:
        try:
            res = session.run(cypher, nid=int(node_id))
            rows = list(res)
            if not rows:
                return []
            entry = rows[0]
            nodes = entry["nodes"]
            rels = entry["rels"]
            # simple textual serialization
            bullets = []
            for n in nodes:
                lbls = ",".join(n["labels"])
                #name = n["props"].get("name") or n["props"].get("description") or ""
                # Format all properties as key: value pairs
                props = n["props"]
                prop_strs = [f"{k}: {v}" for k, v in props.items()]
                prop_text = ", ".join(prop_strs) if prop_strs else "no properties"
                bullets.append(f"Node {n['id']} [{lbls}] - {prop_text}")
                #bullets.append(f"Node {n['id']} [{lbls}] : {name}")
            for r in rels:
                #bullets.append(f"Rel: {r['start']} -[{r['type']}]-> {r['end']}")
                for r in rels:
                    props = r.get("props", {})
                    prop_strs = [f"{k}: {v}" for k, v in props.items()]
                    prop_text = f" ({', '.join(prop_strs)})" if prop_strs else ""
                    bullets.append(f"Rel: {r['start']} -[{r['type']}]{prop_text}-> {r['end']}")
            return bullets
        except Exception as e:
            # fallback if APOC not available: basic 1-hop
            #this should never happen is just to make the code more robust
            with driver.session() as s2:
                q = """
                MATCH (n)-[r]-(m)
                WHERE id(n) = $nid
                RETURN id(n) as n_id, labels(n) as n_labels, properties(n) as n_props,
                       id(m) as m_id, labels(m) as m_labels, properties(m) as m_props, type(r) as rel_type
                LIMIT 100
                """
                rows2 = s2.run(q, nid=int(node_id))
                bullets = []
                for row in rows2:
                    bullets.append(f"Node {row['n_id']} [{','.join(row['n_labels'])}]")
                    bullets.append(f" -[{row['rel_type']}]-> Node {row['m_id']} [{','.join(row['m_labels'])}]")
                return bullets


# build index (run once per graph)

def build_index_main():
    driver = connect_neo4j()
    nodes, edges = fetch_graph(driver)
    print(f"Fetched {len(nodes)} nodes and {len(edges)} edges from Neo4j.")
    node2v = compute_node2vec_embeddings(driver)
    text_emb = compute_text_embeddings(nodes)
    idx_path, meta_path = build_and_store_vector_index(nodes, node2v, text_emb, index_path="faiss_index")
    driver.close()
    return idx_path, meta_path


# Query pipeline (use after index is built)

def query_pipeline(query, index_path="faiss_index/faiss.idx", meta_path="faiss_index/meta.pkl", top_k=5, expand_hops=2):
    # embed query (semantic portion only, structural left as zeros for now)
    q_vec = embed_query_semantic(query)
    hits = load_index_and_search(index_path, meta_path, q_vec, top_k=top_k)
    print("Retriever hits:")
    for h in hits:
        print(h["score"], h["id"], h["meta"]["text"][:120])
    # Expand via cypher
    driver = connect_neo4j()
    contexts = []
    for h in hits:
        nid = h["id"]
        ctx_bullets = cypher_expand_context(driver, nid, hops=expand_hops)
        contexts.append({"node_id": nid, "bullets": ctx_bullets, "score": h["score"], "meta": h["meta"]})
    driver.close()
    # Format final context
    ctx_text_parts = []
    for c in contexts:
        ctx_text_parts.append(f"=== Node {c['node_id']} (score={c['score']}) ===")
        ctx_text_parts.extend(c["bullets"])
    full_context = "\n".join(ctx_text_parts)
    # create prompt for the LLM
    prompt = ChatPromptTemplate.from_template("""
    You are an expert data provenance assistant.
    Use the provided graph context to answer the question **factually and precisely**.
    keep the answer short.
    If the answer is not explicitly supported by the context, reply “I don’t know”.

    Question:
    {question}

    Graph context:
    {context}
    """)
    # LLM
    llm = ChatGroq(
        temperature=0,
        groq_api_key=os.environ.get("GROQ_API_KEY", ""),  # or paste
        model_name="llama-3.3-70b-versatile",
        max_tokens=GEN_MAX_TOKENS
    )

    chain = prompt | llm
    answer = chain.invoke({"question": query, "context": full_context})

    return answer.content, full_context



In [ ]:
# ---------------------------
# Example usage
# ---------------------------
# 1) Build index once per graph
build_index_main()



Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function. ('id' is no longer supported)} {position: line: 1, column: 18, offset: 17} for query: 'MATCH (n) RETURN id(n) as id, labels(n) as labels, properties(n) as props'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function. ('id' is no longer supported)} {position: line: 1, column: 27, offset: 26} for query: 'MATCH (a)-[r]->(b) RETURN id(a) as a, id(b) as b, type(r) as t'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {c

Fetched 31 nodes and 53 edges from Neo4j.
Cleaned up node2vec_emb property from Neo4j.
Saved index and metadata to faiss_index


('faiss_index\\faiss.idx', 'faiss_index\\meta.pkl')

In [18]:
# 2) Query (after building index)
q = "Which columns were affected by encoding steps?"
ans, ctx = query_pipeline(q, index_path="faiss_index/faiss.idx", meta_path="faiss_index/meta.pkl", top_k=5, expand_hops=2)
print("\n=== GENERATED ANSWER ===\n")
print(ans)
print("\n=== USED CONTEXT ===\n")
print(ctx)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function. ('id' is no longer supported)} {position: line: 3, column: 15, offset: 25} for query: '\n    MATCH (n)\n    WHERE id(n) = $nid\n    CALL apoc.path.subgraphAll(n, {maxLevel: 2})\n    YIELD nodes, relationships\n    RETURN [x IN nodes | {id: id(x), labels: labels(x), props: properties(x)}] as nodes,\n           [r IN relationships | {start: id(startNode(r)), end: id(endNode(r)), type: type(r)}] as rels\n    '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function. ('id' is no longer supported)} 

Retriever hits:
0.30932238698005676 5 Emb_trans = LabelEncoder()
df["Embarked"] = sex_trans.fit_transform(df["Embarked"]) | Use a LabelEncoder to transform a 
0.2711818218231201 4 sex_trans = LabelEncoder()
df["Sex"] = sex_trans.fit_transform(df["Sex"]) | Use a LabelEncoder to transform a categorica
0.22886817157268524 30 Parch | [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 0, 2, 2, 
0.22168490290641785 21 Embarked | [1, 2, 0, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 0, 2, 2, 0, 2, 2, 0, 2, 2, 2, 2, 2, 
0.2205532193183899 25 SibSp | [0, 0, 0, 1, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 3, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 

=== GENERATED ANSWER ===

The columns affected by encoding steps are: 
Embarked (Node 18, 21), 
Sex (Node 20, 22).

=== USED CONTEXT ===

=== Node 5 (score=0.30932238698005676) ===
Node 5 [Activity] : 
Node 6 [Activity] : 
Node 4 [Activity] : 
Node 18 [Column] : Emba

In [2]:
#esempio test column/activity focus
queries_CA=["Which columns were removed during preprocessing?",
            "What operation was applied to the Sex column?",
            "Which columns were transformed into numerical values?",
            "What activity modified the ID column’s data type?",
            "After which activity was the ID column moved to the last position in the dataset?"]

In [3]:
# 2) Query (after building index)
for q in queries_CA:
    print("\n=== USER QUESTION ===\n")
    print(q)
    ans, ctx = query_pipeline(q, index_path="faiss_index/faiss.idx", meta_path="faiss_index/meta.pkl", top_k=5, expand_hops=2)
    print("\n=== GENERATED ANSWER ===\n")
    print(ans)


=== USER QUESTION ===

Which columns were removed during preprocessing?
Retriever hits:
0.211553156375885 30 Parch | [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 0, 2, 2, 
0.20385104417800903 17 MissAge | [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0
0.1948685646057129 21 Embarked | [1, 2, 0, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 0, 2, 2, 0, 2, 2, 0, 2, 2, 2, 2, 2, 
0.18728891015052795 16 ID | [412, 884, 311, 782, 724, 815, 410, 626, 107, 778, 141, 570, 811, 46, 133, 6, 100, 252, 444, 791, 696, 535, 64, 369
0.18510782718658447 12 PassengerId | [412, 884, 311, 782, 724, 815, 410, 626, 107, 778, 141, 570, 811, 46, 133, 6, 100, 252, 444, 791, 696, 535

=== GENERATED ANSWER ===

The columns removed during preprocessing were "Name", "Ticket", "Cabin", and "Survived".

=== USER QUESTION ===

What operation was applied to the Sex column?
Retrieve

In [16]:
#esempio test Lineage and Dependency Tracing
queries_L=["From which original column does the transformed Sex numeric column originate?",
           "Which steps affected the values in the Age column before it was used to create MissAge?",
           "What transformations directly or indirectly influenced the Embarked column in the final dataset?",
           "Trace the lineage of the column MissAge: which operations and columns contributed to it?",
           "Which activities were performed right before the dataset was split into training and testing sets?"]

In [17]:
for q in queries_L:
    print("\n=== USER QUESTION ===\n")
    print(q)
    ans, ctx = query_pipeline(q, index_path="faiss_index/faiss.idx", meta_path="faiss_index/meta.pkl", top_k=5, expand_hops=2)
    print("\n=== GENERATED ANSWER ===\n")
    print(ans)


=== USER QUESTION ===

From which original column does the transformed Sex numeric column originate?
Retriever hits:
0.4232756197452545 4 sex_trans = LabelEncoder()
df["Sex"] = sex_trans.fit_transform(df["Sex"]) | Use a LabelEncoder to transform a categorica
0.39851927757263184 22 Sex | [1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1,
0.37379205226898193 5 Emb_trans = LabelEncoder()
df["Embarked"] = sex_trans.fit_transform(df["Embarked"]) | Use a LabelEncoder to transform a 
0.2783627212047577 19 Age | [0.0, 28.0, 24.0, 17.0, 50.0, 30.5, 0.0, 61.0, 21.0, 5.0, 0.0, 32.0, 26.0, 0.0, 47.0, 0.0, 34.0, 29.0, 28.0, 0.0, 
0.26487189531326294 30 Parch | [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 0, 2, 2, 

=== GENERATED ANSWER ===

The transformed Sex numeric column originates from the original 'Sex' column.

=== USER QUESTION ===

Which steps affected the values in th

In [11]:
#esempio test Semantic / Descriptive Reasoning
queries_S=["Which operations can be categorized as encoding steps in this pipeline?",
           "Which activities modified the dataset structure (i.e., changed the order or presence of columns)?"]

In [15]:
for q in queries_S:
    print("\n=== USER QUESTION ===\n")
    print(q)
    ans, ctx = query_pipeline(q, index_path="faiss_index/faiss.idx", meta_path="faiss_index/meta.pkl", top_k=5, expand_hops=2)
    print("\n=== GENERATED ANSWER ===\n")
    print(ans)


=== USER QUESTION ===

Which operations can be categorized as encoding steps in this pipeline?
Retriever hits:
0.23726511001586914 18 Embarked | ['Q', 'S', 'C', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'C', 'S', 'S', 'S', 'S', 'Q', 'S', 'S', 'S', 'Q', 'S', 'S'
0.23260673880577087 10 y = to_categorical(y.values, num_classes=2) | Transform the target variable into a one-hot encoded format | One-hot enco
0.1718009114265442 5 Emb_trans = LabelEncoder()
df["Embarked"] = sex_trans.fit_transform(df["Embarked"]) | Use a LabelEncoder to transform a 
0.1610213816165924 15 Cabin | [nan, nan, 'C54', 'B20', nan, nan, nan, 'D50', nan, nan, nan, nan, nan, nan, nan, nan, nan, 'G6', nan, nan, nan,
0.14721859991550446 4 sex_trans = LabelEncoder()
df["Sex"] = sex_trans.fit_transform(df["Sex"]) | Use a LabelEncoder to transform a categorica

=== GENERATED ANSWER ===

The operations that can be categorized as encoding steps in this pipeline are:

1. Transforming the 'Embarked' column into numerical values using

In [5]:
q="how many activities are there?"
print("\n=== USER QUESTION ===\n")
print(q)
ans, ctx = query_pipeline(q, index_path="faiss_index/faiss.idx", meta_path="faiss_index/meta.pkl", top_k=5, expand_hops=2)
print("\n=== GENERATED ANSWER ===\n")
print(ans)


=== USER QUESTION ===

how many activities are there?
Retriever hits:
0.29186636209487915 8 df['ID'] = df['ID'] / 1e7 | Perform a mathematical operation on all values in a column | Divide all values in the 'ID' c
0.22890862822532654 9 df = df[[col for col in df.columns if col != 'ID'] + ['ID']] | Reorder the columns in the DataFrame | Move the 'ID' colu
0.2272200584411621 2 df = df.dropna(subset=["Embarked"]) | Remove rows from the DataFrame where a specific column has missing values | Drop r
0.21978066861629486 1 df = df.drop(columns=["Name", "Ticket", "Cabin"]) | Remove columns from the DataFrame that might not be useful for the a
0.21819862723350525 10 y = to_categorical(y.values, num_classes=2) | Transform the target variable into a one-hot encoded format | One-hot enco

=== GENERATED ANSWER ===

There are 11 activities:

1. Node 8: Divide all values in the 'ID' column by 1e7
2. Node 7: Convert the 'ID' column to float32 data type
3. Node 9: Move the 'ID' column to the last positi

# 2a versione

In [ ]:
import os
import numpy as np
from neo4j import GraphDatabase
import networkx as nx
from sentence_transformers import SentenceTransformer
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from sklearn.preprocessing import normalize
import pickle
import faiss


# Configuration
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = os.getenv("NEO4J_PASS", "adminadmin")
# node properties with textual info for embedding
TEXT_FIELDS = ["code", "context", "feature_name","function_name","instance","name","runtime_exceptions","type","value","id"]


# Node2Vec params
NODE2VEC_DIM = 128
NODE2VEC_P = 1
NODE2VEC_Q = 1
WALK_LENGTH = 30
NUM_WALKS = 10

# Text embedding model
TEXT_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2" 

GEN_MAX_TOKENS = 256




def connect_neo4j():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS), notifications_disabled_categories=["DEPRECATION"])
    return driver

def fetch_graph(driver):
    """
    Fetch nodes and relationships from Neo4j.
    Expects nodes to have an 'id' property or uses internal id.
    Returns:
        nodes: {node_id: {labels: [...], props: {...}}}
        edges: list of (src_id, dst_id, rel_type)
    """
    with driver.session() as session:
        # fetch nodes with their internal id, labels and properties
        nodes_res = session.run("MATCH (n) RETURN id(n) as id, labels(n) as labels, properties(n) as props")
        nodes = {}
        for r in nodes_res:
            nid = r["id"]
            nodes[nid] = {"labels": r["labels"], "props": r["props"]}
        # fetch relationships
        rels_res = session.run("MATCH (a)-[r]->(b) RETURN id(a) as a, id(b) as b, type(r) as t")
        edges = []
        for r in rels_res:
            edges.append((r["a"], r["b"], r["t"]))
    return nodes, edges


# Build structural embeddings via node2vec

def compute_node2vec_embeddings(driver, dim=NODE2VEC_DIM):
    """
    Compute structural embeddings using Neo4j Graph Data Science (GDS) node2vec.
    Stores embeddings back into Neo4j and returns a dict {node_id: np.array}.
    """

    # Create a GDS in-memory projection
    with driver.session() as session:
        # ensures the projection doesn't already exist from a failed run
        session.run("CALL gds.graph.drop('provGraph', false)")
        #if not creates it
        session.run("""
        CALL gds.graph.project(
            'provGraph',
            '*',          // all node labels
            '*'           // all relationship types
        )
        """)

    # Run node2vec and WRITE the results back to Neo4j 
    # this is the "structural" part of the embedding
    with driver.session() as session:
        # Use gds.node2vec.write (does the same thing as mutate but also allows saving)
        session.run(f"""
        CALL gds.node2vec.write('provGraph', {{
            embeddingDimension: {dim},
            writeProperty: 'node2vec_emb',  // specify the property name to save
            walkLength: 40,
            iterations: 5,
            inOutFactor: 1.0,
            returnFactor: 1.0,
            windowSize: 5,
            concurrency: 2
        }})
        YIELD nodePropertiesWritten, computeMillis
        """)

    # Extract embeddings back into Python
    embeddings = {}
    with driver.session() as session:
        result = session.run("MATCH (n) WHERE n.node2vec_emb IS NOT NULL RETURN id(n) as id, n.node2vec_emb as emb")
        for record in result:
            nid = record["id"]
            emb = np.array(record["emb"], dtype=np.float32)
            embeddings[nid] = emb

    # Drop the in-memory projection to free resources
    with driver.session() as session:
        session.run("CALL gds.graph.drop('provGraph', false)")
    
    # Clean up the property from your graph
    with driver.session() as session:
        session.run("MATCH (n) WHERE n.node2vec_emb IS NOT NULL REMOVE n.node2vec_emb")
        print("Cleaned up node2vec_emb property from Neo4j.")

    return embeddings



# Build text embeddings

def compute_text_embeddings(nodes, model_name=TEXT_EMBED_MODEL):
    st = SentenceTransformer(model_name)
    text_emb = {}
    for nid, node in nodes.items():
        # build a textual summary from selected properties
        parts = []
        for f in TEXT_FIELDS:
            if f in node["props"] and node["props"][f]:
                parts.append(str(node["props"][f]))
        # include labels as short context
        parts.append("labels: " + ",".join(node["labels"]))
        text = " | ".join(parts) if parts else f"node_{nid}"
        vec = st.encode(text)
        text_emb[nid] = {"vec": vec, "text": text}
    return text_emb



#  build_and_store_vector_indexes (separate indexes for structural and semantic THIS IS THE MAIN DIFFERENCE FROM v1)

def build_and_store_vector_indexes(nodes, node2vec_emb, text_emb, index_path="faiss_indexes"):
    """
    Builds and stores two separate FAISS indexes: one for semantic vectors and one for structural vectors.
    """
    ids = []
    semantic_vectors = []
    structural_vectors = []
    metadatas = []

    # Use a common list of node IDs for both indexes
    valid_node_ids = sorted(list(node2vec_emb.keys()))

    for nid in valid_node_ids:
        # Ensure text embedding also exists, though it should if node2vec does
        if nid not in text_emb:
            continue

        s_vec = node2vec_emb[nid]
        t_vec = text_emb[nid]["vec"]

        # Normalize each vector individually
        s_vec_norm = normalize(s_vec.reshape(1, -1), norm='l2')[0]
        t_vec_norm = normalize(t_vec.reshape(1, -1), norm='l2')[0]

        ids.append(str(nid))
        structural_vectors.append(s_vec_norm.astype("float32"))
        semantic_vectors.append(t_vec_norm.astype("float32"))
        metadatas.append({
            "node_id": str(nid),
            "text": text_emb[nid]["text"],
            "labels": nodes[nid]["labels"],
            "props": nodes[nid]["props"]
        })

    os.makedirs(index_path, exist_ok=True)

    # --- Create and save Semantic Index ---
    sem_xb = np.vstack(semantic_vectors)
    sem_dim = sem_xb.shape[1]
    semantic_index = faiss.IndexFlatIP(sem_dim)
    # The vectors are already normalized, no need to call faiss.normalize_L2
    semantic_index.add(sem_xb)
    faiss.write_index(semantic_index, os.path.join(index_path, "semantic.idx"))
    print(f"Saved semantic index to {index_path}/semantic.idx")

    # --- Create and save Structural Index ---
    struct_xb = np.vstack(structural_vectors)
    struct_dim = struct_xb.shape[1]
    structural_index = faiss.IndexFlatIP(struct_dim)
    structural_index.add(struct_xb)
    faiss.write_index(structural_index, os.path.join(index_path, "structural.idx"))
    print(f"Saved structural index to {index_path}/structural.idx")

    # --- Store metadata and the raw vectors for easy lookup ---
    meta_path = os.path.join(index_path, "meta.pkl")
    with open(meta_path, "wb") as f:
        pickle.dump({
            "ids": ids, 
            "metadatas": metadatas,
            "structural_vectors": struct_xb, # Save for lookup
            "semantic_vectors": sem_xb, # Save for lookup
        }, f)
    print(f"Saved metadata and vectors to {meta_path}")
    
    return index_path, meta_path


# Load index + perform similarity search (nearest neighbors via FAISS)

def load_index_and_search(index_path, meta_path, query_vec, top_k=5):
    index = faiss.read_index(index_path)
    # load metadata
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)
    # normalize query
    q = query_vec.reshape(1, -1).astype("float32")
    faiss.normalize_L2(q)
    D, I = index.search(q, top_k) #top k most similar verctors
    hits = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        hits.append({"score": float(score), "id": meta["ids"][idx], "meta": meta["metadatas"][idx]})
    return hits





# Cypher expansion: for each retrieved node, expand (2 but can be customized) hops to collect provenance context

def cypher_expand_context(driver, node_id, hops=2, include_types=None):
    """
    Returns a list of paths (as textual bullet points) around the node.
    """
    include_clause = ""
    if include_types:
        #customization
        # e.g. only include Activity nodes etc. Not enforced here; TALK ABOUT POSSIBILITIES NEXT UPDATE MEETING
        pass
    cypher = f"""
    MATCH (n)
    WHERE id(n) = $nid
    CALL apoc.path.subgraphAll(n, {{maxLevel: {hops}}})
    YIELD nodes, relationships
    RETURN [x IN nodes | {{id: id(x), labels: labels(x), props: properties(x)}}] as nodes,
           [r IN relationships | {{start: id(startNode(r)), end: id(endNode(r)), type: type(r)}}] as rels
    """
    # Uses APOC to call the query and save the bullet points(text)
    with driver.session() as session:
        try:
            res = session.run(cypher, nid=int(node_id))
            rows = list(res)
            if not rows:
                return []
            entry = rows[0]
            nodes = entry["nodes"]
            rels = entry["rels"]
            # simple textual serialization
            bullets = []
            for n in nodes:
                lbls = ",".join(n["labels"])
                #name = n["props"].get("name") or n["props"].get("description") or ""
                # Format all properties as key: value pairs
                props = n["props"]
                prop_strs = [f"{k}: {v}" for k, v in props.items()]
                prop_text = ", ".join(prop_strs) if prop_strs else "no properties"
                bullets.append(f"Node {n['id']} [{lbls}] - {prop_text}")
                #bullets.append(f"Node {n['id']} [{lbls}] : {name}")
            for r in rels:
                #bullets.append(f"Rel: {r['start']} -[{r['type']}]-> {r['end']}")
                for r in rels:
                    props = r.get("props", {})
                    prop_strs = [f"{k}: {v}" for k, v in props.items()]
                    prop_text = f" ({', '.join(prop_strs)})" if prop_strs else ""
                    bullets.append(f"Rel: {r['start']} -[{r['type']}]{prop_text}-> {r['end']}")
            return bullets
        except Exception as e:
            # fallback if APOC not available: basic 1-hop
            #this should never happen is just to make the code more robust
            with driver.session() as s2:
                q = """
                MATCH (n)-[r]-(m)
                WHERE id(n) = $nid
                RETURN id(n) as n_id, labels(n) as n_labels, properties(n) as n_props,
                       id(m) as m_id, labels(m) as m_labels, properties(m) as m_props, type(r) as rel_type
                LIMIT 100
                """
                rows2 = s2.run(q, nid=int(node_id))
                bullets = []
                for row in rows2:
                    bullets.append(f"Node {row['n_id']} [{','.join(row['n_labels'])}]")
                    bullets.append(f" -[{row['rel_type']}]-> Node {row['m_id']} [{','.join(row['m_labels'])}]")
                return bullets


# build index (run once per graph)

def build_index_main():
    driver = connect_neo4j()
    nodes, edges = fetch_graph(driver)
    print(f"Fetched {len(nodes)} nodes and {len(edges)} edges from Neo4j.")
    node2v = compute_node2vec_embeddings(driver)
    text_emb = compute_text_embeddings(nodes)
    # Call the updated function
    idx_path, meta_path = build_and_store_vector_indexes(nodes, node2v, text_emb, index_path="faiss_indexes")
    driver.close()
    return idx_path, meta_path


# Query pipeline (use after index is built)

def query_pipeline(query, 
                    index_path="faiss_indexes", 
                    meta_path="faiss_indexes/meta.pkl", 
                    top_k=5, 
                    expand_hops=2):
    
    # --- Load Indexes and Metadata ---
    semantic_index = faiss.read_index(os.path.join(index_path, "semantic.idx"))
    structural_index = faiss.read_index(os.path.join(index_path, "structural.idx"))
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)
    
    # Create a map from node_id to its index for easy lookup
    id_to_idx = {node_id: i for i, node_id in enumerate(meta["ids"])}

    # --- Stage 1: Pivot Search (Semantic) ---
    st = SentenceTransformer(TEXT_EMBED_MODEL)
    query_sem_vec = st.encode(query)
    q_vec = normalize(query_sem_vec.reshape(1, -1), norm='l2').astype("float32")

    D_sem, I_sem = semantic_index.search(q_vec, top_k)
    
    pivot_hits = []
    #print("\n--- Stage 1: Semantic Pivot Hits ---") #used for debgging 
    for score, idx in zip(D_sem[0], I_sem[0]):
        if idx < 0: continue
        node_id = meta["ids"][idx]
        pivot_hits.append({'id': node_id, 'score': score, 'idx': idx})
        #print(f"ID: {node_id}, Score: {score:.4f}, Text: {meta['metadatas'][idx]['text'][:100]}") #used for debgging 

    # --- Stage 2: Candidate Expansion (Structural) ---
    candidate_nodes = {} # Use a dict to store best score for each node_id

    #print("\n--- Stage 2: Structural Candidate Expansion ---") #used for debgging 
    for hit in pivot_hits:
        # Add the pivot node itself to the candidates
        candidate_nodes[hit['id']] = candidate_nodes.get(hit['id'], 0) + hit['score']
        
        # Get the structural vector of the pivot node
        struct_vec_of_pivot = meta['structural_vectors'][hit['idx']].reshape(1, -1)
        
        # Search for structurally similar nodes
        D_struct, I_struct = structural_index.search(struct_vec_of_pivot, top_k)
        
        #print(f"  Expanding from pivot {hit['id']}:") #used for debgging 
        for score, idx in zip(D_struct[0], I_struct[0]):
            if idx < 0: continue
            node_id = meta["ids"][idx]
            # Add structural score to the candidate's total score
            # The initial pivot gets a semantic score. The expansion adds structural scores.
            semantic_weight = 0.9
            structural_weight = 0.1
            
            # Initialize with semantic score if it was a pivot, otherwise 0
            current_score = candidate_nodes.get(node_id, 0)
            if node_id == hit['id']: # It's the pivot
                 candidate_nodes[node_id] = (hit['score'] * semantic_weight)
            else: # It's a structurally similar node
                 candidate_nodes[node_id] = current_score + (score * structural_weight)

            #print(f"    - Found candidate {node_id} (structural score: {score:.4f})") #used for debgging 
    
    # --- Stage 3: Re-ranking and Consolidation ---
    # Sort candidates by their combined score in descending order
    sorted_candidates = sorted(candidate_nodes.items(), key=lambda item: item[1], reverse=True)
    
    final_hits = []
    #print(f"\n--- Final Top-{top_k} Hits after Re-ranking ---") #used for debgging 
    for node_id, final_score in sorted_candidates[:top_k]:
        idx = id_to_idx[node_id]
        final_hits.append({"id": node_id, "score": final_score, "meta": meta["metadatas"][idx]})
        # print(f"ID: {node_id}, Final Score: {final_score:.4f}, Text: {meta['metadatas'][idx]['text'][:100]}")  #used for debgging 
        
    # --- Stage 4: Context Generation & LLM ---
    driver = connect_neo4j()
    contexts = []
    for h in final_hits:
        nid = h["id"]
        ctx_bullets = cypher_expand_context(driver, nid, hops=expand_hops)
        contexts.append({"node_id": nid, "bullets": ctx_bullets, "score": h["score"], "meta": h["meta"]})
    driver.close()
    
    ctx_text_parts = []
    for c in contexts:
        ctx_text_parts.append(f"=== Node {c['node_id']} (relevance score={c['score']:.4f}) ===")
        ctx_text_parts.extend(c["bullets"])
    full_context = "\n".join(ctx_text_parts)
    # create prompt for the LLM
    prompt = ChatPromptTemplate.from_template("""
    You are an expert data provenance assistant.
    Use the provided graph context to answer the question **factually and precisely**.
    keep the answer short.
    If the answer is not explicitly supported by the context, reply “I don’t know”.

    Question:
    {question}

    Graph context:
    {context}
    """)
    # LLM
    llm = ChatGroq(
        temperature=0,
        groq_api_key=os.environ.get("GROQ_API_KEY", ""),  # or paste
        model_name="llama-3.3-70b-versatile",
        max_tokens=GEN_MAX_TOKENS
    )

    chain = prompt | llm
    answer = chain.invoke({"question": query, "context": full_context})

    return answer.content, full_context



In [ ]:
build_index_main() 

Fetched 26 nodes and 43 edges from Neo4j.
Cleaned up node2vec_emb property from Neo4j.
Saved semantic index to faiss_indexes/semantic.idx
Saved structural index to faiss_indexes/structural.idx
Saved metadata and vectors to faiss_indexes\meta.pkl


('faiss_indexes', 'faiss_indexes\\meta.pkl')

In [8]:
q="how many activities are there?"
print("\n=== USER QUESTION ===\n")
print(q)
ans, ctx = query_pipeline(q, index_path="faiss_indexes", meta_path="faiss_indexes/meta.pkl", top_k=5, expand_hops=2)
print("\n=== GENERATED ANSWER ===\n")
print(ans)


=== USER QUESTION ===

how many activities are there?

--- Stage 1: Semantic Pivot Hits ---
ID: 8, Score: 0.3499, Text: df['ID'] = df['ID'] / 1e7 | Perform an arithmetic operation on a column in the DataFrame | Divide al
ID: 2, Score: 0.2768, Text: df = df.dropna(subset=["Embarked"]) | Remove rows from the DataFrame where a specific column has mis
ID: 24, Score: 0.2661, Text: Age | [0.0, 30.0, 26.0, 49.0, 43.0, 36.0, 0.0, 26.0, 39.0, 0.0, 22.0, 36.0, 5.0, 19.0, 0.75, 25.0, 1
ID: 10, Score: 0.2654, Text: y = to_categorical(y.values, num_classes=2) | One-hot encode a categorical target variable | One-hot
ID: 9, Score: 0.2479, Text: df = df[[col for col in df.columns if col != 'ID'] + ['ID']] | Reorder the columns in the DataFrame 

--- Stage 2: Structural Candidate Expansion ---
  Expanding from pivot 8:
    - Found candidate 8 (structural score: 1.0000)
    - Found candidate 9 (structural score: 0.9999)
    - Found candidate 20 (structural score: 0.9998)
    - Found candidate 21 (struc

In [10]:
#esempio test column/activity focus
queries_CA=["Which columns were removed during preprocessing?",
            "What operation was applied to the Sex column?",
            "Which columns were transformed into numerical values?",
            "What activity modified the ID column’s data type?",
            "After which activity was the ID column moved to the last position in the dataset?"]

In [13]:
# 2) Query (after building index)
for q in queries_CA:
    print("\n=== USER QUESTION ===\n")
    print(q)
    ans, ctx = query_pipeline(q, index_path="faiss_indexes", meta_path="faiss_indexes/meta.pkl", top_k=5, expand_hops=2)
    print("\n=== GENERATED ANSWER ===\n")
    print(ans)


=== USER QUESTION ===

Which columns were removed during preprocessing?

=== GENERATED ANSWER ===

The columns removed during preprocessing are 'Name', 'Ticket', and 'Cabin' (Node 1 [Activity]) and 'Survived' (Node 6 [Activity]).

=== USER QUESTION ===

What operation was applied to the Sex column?

=== GENERATED ANSWER ===

The operation applied to the Sex column was Label Encoding, which transformed the categorical values into numerical values.

=== USER QUESTION ===

Which columns were transformed into numerical values?

=== GENERATED ANSWER ===

The columns 'Embarked' and 'Sex' were transformed into numerical values using LabelEncoder.

=== USER QUESTION ===

What activity modified the ID column’s data type?

=== GENERATED ANSWER ===

The activity that modified the ID column's data type is Node 7, with the code `df['ID'] = df['ID'].astype(np.float32)`.

=== USER QUESTION ===

After which activity was the ID column moved to the last position in the dataset?

=== GENERATED ANSWER ==

In [ ]:
project/    
├── rag_system/
│     ├── config/
│     │   ├── __init__.py
│    │   └── settings.py              # Configuration (model names, weights, paths)
│    │
│    ├── embeddings/
│    │   ├── __init__.py
│    │   ├── semantic_embedder.py     # mini-L6-v2 embedding logic
│    │   └── structural_embedder.py   # Node2Vec embedding logic
│    │
│    ├── indexing/
│    │   ├── __init__.py
│    │   ├── faiss_indexer.py         # FAISS index creation/management
│    │   └── graph_indexer.py         # Orchestrates indexing pipeline
│    │
│    ├── retrieval/
│    │   ├── __init__.py
│    │   ├── semantic_retriever.py    # Semantic search (top-5)
│    │   ├── structural_expander.py   # Structural neighbor expansion
│    │   ├── score_combiner.py        # Weighted scoring logic
│    │   └── graph_expander.py        # 2-hop Cypher expansion
│    │
│    ├── preprocessing/
│    │   ├── __init__.py
│    │   └── context_formatter.py     # Convert nodes to bullet points
│    │
│    ├── llm/
│    │   ├── __init__.py
│    │   └── generator.py             # LLM interface/ gives prompt and functions to save/load history
│    │
│    ├── database/
│    │   ├── __init__.py
│    │   └── neo4j_client.py          # Neo4j connection/queries
│    │
│    ├── rag_pipeline.py              # Main RAG orchestrator and generates answer/update chat history
├── streamlit_app.py      #my app that will use the rag
├── query_rag.py          #file that is used by the app to call the rag_pipeline      